# Decision Trees — Exercises

Companion to the note [Decision Trees](Decision%20Trees.md).

Practise impurity measures (entropy, Gini, variance), information gain and gain ratio, greedy split search, recursive CART, cost-complexity pruning, and the two weak spots of trees (instability and axis-aligned boundaries), by hand and from scratch in NumPy.

**17 exercises** · 🧠 Concept ×4 · ✍️ By hand ×6 · 💻 Code ×7

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from scipy.stats import entropy as sp_entropy
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.model_selection import cross_val_score
from sklearn.datasets import load_breast_cancer, load_iris
rng = np.random.default_rng(0)
Xbc, ybc = load_breast_cancer(return_X_y=True)
perm = rng.permutation(len(ybc))
tr, te = perm[:400], perm[400:]
Xtr, ytr, Xte, yte = Xbc[tr], ybc[tr], Xbc[te], ybc[te]

## Part A · Concepts

### Exercise 1 · Greedy, not optimal
*🧠 Concept · ★☆☆*

CART picks, at each node, the single split that most reduces impurity, and never revisits it. Why not search for the globally best tree?
Give a small example where the greedy choice is bad (hint: XOR).

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

On XOR data, what is the impurity decrease of any single axis-aligned split at the root?

</details>

<details>
<summary><b>✅ Solution</b></summary>

Finding the smallest/most accurate tree is NP-hard (exponentially many trees), so greedy top-down search is used. On XOR ($y=x_1\oplus x_2$ on a grid),
every split on $x_1$ or $x_2$ at the root leaves each child 50/50, so the gain is $0$, the same as a useless split. A greedy learner with a
'stop if gain = 0' rule stops immediately, although a depth-2 tree classifies XOR perfectly. In practice we grow deep and then prune.

</details>

### Exercise 2 · Why not misclassification error?
*🧠 Concept · ★★☆*

A parent node has 400 points of class A and 400 of class B. Split 1 gives children (300A,100B) and (100A,300B). Split 2 gives (200A,400B) and (200A,0B).
Compute the weighted misclassification error and the weighted Gini of both splits. Which split does each criterion prefer, and why are Gini/entropy used to grow trees?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Misclassification error of a node: $1-\max_kp_k$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Misclassification: split 1 $=\tfrac12\cdot\tfrac14+\tfrac12\cdot\tfrac14=0.25$; split 2 $=\tfrac34\cdot\tfrac13+\tfrac14\cdot0=0.25$: a tie.
Gini: split 1 $=2\cdot\tfrac12\cdot(1-\tfrac{9}{16}-\tfrac1{16})=0.375$; split 2 $=\tfrac34\cdot\tfrac49+\tfrac14\cdot0\approx0.333$, so Gini prefers split 2, which creates a **pure** node that needs no further work.
Gini and entropy are strictly concave in $p$, so they reward purer children even when the error count does not change; misclassification error is piecewise linear and often insensitive.
It is still a good criterion for *pruning*.

</details>

### Exercise 3 · Gain ratio and many-valued attributes
*🧠 Concept · ★★☆*

Why would ID3 (plain information gain) love a 'customer ID' attribute? How does C4.5's gain ratio $\frac{\text{gain}}{\text{SplitInfo}}$ with
$\text{SplitInfo}=-\sum_v\frac{|S_v|}{|S|}\log_2\frac{|S_v|}{|S|}$ counter this? Why does CART not have this problem in the same form?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

A unique ID splits $N$ points into $N$ pure children.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Splitting on a unique ID gives $N$ pure leaves, so the gain equals the full parent entropy: the maximum possible. It is useless for generalisation.
SplitInfo of that split is $\log_2N$ (large), so the ratio is penalised; attributes with few balanced values have small SplitInfo.
CART only makes **binary** splits, so a many-valued attribute cannot shatter the data in one step (though it still has more candidate splits, a milder bias).

</details>

### Exercise 4 · Strengths and weaknesses
*🧠 Concept · ★☆☆*

For each property from the note, give a one-line reason: (a) no feature scaling needed, (b) handles mixed feature types,
(c) unstable / high variance, (d) axis-aligned boundaries. How do [[Ensemble Methods]] address (c)?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

A split is a threshold on a single feature. What does a monotone rescaling do to the ordering of values?

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) Splits only depend on the *order* of values within one feature; any monotone transform leaves the tree unchanged.
(b) Numeric features use thresholds, categorical ones use subsets of categories; no distances between features are ever computed.
(c) A small change in the data can change the root split, and then the whole subtree below differs (Exercise 16).
(d) Each split is perpendicular to an axis, so a diagonal boundary needs a staircase of many splits (Exercise 17).
Averaging many trees (bagging/random forests) reduces variance; boosting shallow trees reduces bias.

</details>

## Part B · By hand

### Exercise 5 · Entropy and Gini of a node
*✍️ By hand · ★☆☆*

A node contains 9 positive and 5 negative examples. Compute its entropy (bits) and Gini impurity.

In [ ]:
H_node = None
G_node = None

_p = np.array([9, 5]) / 14
check('entropy', H_node, sp_entropy(_p, base=2), tol=1e-3)
check('Gini', G_node, 1 - np.sum(_p**2), tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

$p=(9/14, 5/14)\approx(0.643, 0.357)$; $\log_2 0.643\approx-0.637$, $\log_2 0.357\approx-1.485$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$H=-0.643\log_20.643-0.357\log_20.357\approx0.410+0.531=0.940$ bits. Gini $=1-0.643^2-0.357^2=1-\frac{81+25}{196}\approx0.459$.

```python
H_node = 0.9403
G_node = 1 - (81 + 25) / 196
```

</details>

### Exercise 6 · Information gain of a split
*✍️ By hand · ★☆☆*

The parent $(9+,5-)$ is split by *Wind* into Weak $(6+,2-)$ and Strong $(3+,3-)$. Compute the information gain (entropy, bits) and the decrease in Gini.

In [ ]:
gain_wind = None
gini_dec_wind = None

_H = lambda c: sp_entropy(np.array(c) / sum(c), base=2)
_G = lambda c: 1 - np.sum((np.array(c) / sum(c))**2)
check('information gain', gain_wind, _H([9,5]) - 8/14*_H([6,2]) - 6/14*_H([3,3]), tol=1e-3)
check('Gini decrease', gini_dec_wind, _G([9,5]) - 8/14*_G([6,2]) - 6/14*_G([3,3]), tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

$H(6,2)=0.811$, $H(3,3)=1$. Gini$(6,2)=0.375$, Gini$(3,3)=0.5$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Gain $=0.940-\tfrac8{14}(0.811)-\tfrac6{14}(1)=0.940-0.464-0.429\approx0.048$ bits.
Gini decrease $=0.459-\tfrac8{14}(0.375)-\tfrac6{14}(0.5)=0.459-0.214-0.214\approx0.031$. A weak split.

```python
gain_wind = 0.0481
gini_dec_wind = 0.0306
```

</details>

### Exercise 7 · Gain ratio
*✍️ By hand · ★★☆*

*Outlook* splits $(9+,5-)$ into Sunny $(2+,3-)$, Overcast $(4+,0-)$, Rain $(3+,2-)$. Compute the information gain, the SplitInfo and the gain ratio.

In [ ]:
gain_outlook = None
split_info = None
gain_ratio = None

_H = lambda c: sp_entropy(np.array(c) / sum(c), base=2)
_g = _H([9,5]) - 5/14*_H([2,3]) - 4/14*_H([4]) - 5/14*_H([3,2])
check('gain', gain_outlook, _g, tol=1e-3); check('SplitInfo', split_info, _H([5,4,5]), tol=1e-3)
check('gain ratio', gain_ratio, _g / _H([5,4,5]), tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

$H(2,3)=H(3,2)=0.971$, $H(4,0)=0$. SplitInfo is the entropy of the child *sizes* $(5,4,5)/14$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Gain $=0.940-\tfrac5{14}0.971-0-\tfrac5{14}0.971\approx0.247$. SplitInfo $=H(5/14,4/14,5/14)\approx1.577$. Gain ratio $\approx0.156$.
Outlook has a much larger gain than Wind (0.048) and would be chosen at the root.

```python
gain_outlook = 0.2467
split_info = 1.5774
gain_ratio = 0.2467 / 1.5774
```

</details>

### Exercise 8 · A regression split by hand
*✍️ By hand · ★☆☆*

Regression data: $x=(1,2,3,4,5,6)$, $y=(1.0,\ 1.2,\ 0.9,\ 3.0,\ 3.2,\ 2.9)$. Using the variance criterion, find the best threshold (midpoint) and the
weighted child variance $\frac{n_L}{n}\operatorname{Var}_L+\frac{n_R}{n}\operatorname{Var}_R$. What does each leaf predict?

In [ ]:
thr_best = None
wvar_best = None
leaf_means = None  # [left, right]

_x = np.arange(1, 7.)[:, None]; _y = np.array([1, 1.2, .9, 3, 3.2, 2.9])
_t = DecisionTreeRegressor(max_depth=1).fit(_x, _y).tree_
check('threshold', thr_best, _t.threshold[0], tol=1e-3)
check('weighted child variance', wvar_best, (_t.n_node_samples[1]*_t.impurity[1] + _t.n_node_samples[2]*_t.impurity[2]) / 6, tol=1e-4)
check('leaf means', leaf_means, _t.value[1:3].ravel(), tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

Look at the data: the jump is between $x=3$ and $x=4$. Each side has mean $1.033$ and $3.033$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Threshold $3.5$. Left $(1.0,1.2,0.9)$: mean $1.0333$, variance $0.01556$; right $(3.0,3.2,2.9)$: mean $3.0333$, variance $0.01556$.
Weighted variance $0.01556$, down from the parent's $1.0156$ (a reduction of exactly $1.0$). The leaves predict $1.033$ and $3.033$.

```python
thr_best = 3.5
wvar_best = 0.015556
leaf_means = [3.1/3, 9.1/3]
```

</details>

### Exercise 9 · Cost-complexity pruning: the weakest link
*✍️ By hand · ★★☆*

Cost-complexity pruning minimises $R_\alpha(T)=R(T)+\alpha|T|$, with $R$ the (training) error rate and $|T|$ the number of leaves.
An internal node $t$ has error $R(t)=0.15$ if collapsed to a leaf; its subtree $T_t$ has 3 leaves with total error $R(T_t)=0.05$.
(a) For which $\alpha$ is collapsing better? Store the critical $\alpha^*$. (b) Do we prune at $\alpha=0.03$? At $\alpha=0.08$?

In [ ]:
alpha_star = None
prune_at_003 = None  # True/False
prune_at_008 = None

check('alpha*', alpha_star, (0.15-0.05)/(3-1))
check('alpha=0.03', prune_at_003, 0.15 + 0.03 <= 0.05 + 3*0.03)
check('alpha=0.08', prune_at_008, 0.15 + 0.08 <= 0.05 + 3*0.08)

<details>
<summary><b>💡 Hint</b></summary>

Collapse when $R(t)+\alpha\cdot1\le R(T_t)+\alpha|T_t|$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$R(t)+\alpha\le R(T_t)+3\alpha\iff\alpha\ge\frac{R(t)-R(T_t)}{|T_t|-1}=\frac{0.10}{2}=0.05$.
At $\alpha=0.03$: $0.18>0.14$, keep the subtree. At $\alpha=0.08$: $0.23\le0.29$, prune. Increasing $\alpha$ prunes the weakest links one at a time,
which gives the nested sequence of trees that `cost_complexity_pruning_path` returns; pick $\alpha$ by cross-validation (Exercise 15).

```python
alpha_star = 0.05
prune_at_003 = False
prune_at_008 = True
```

</details>

### Exercise 10 · Bounds on the impurities
*✍️ By hand · ★★☆*

Show that for $K$ classes the entropy is at most $\log_2K$ and the Gini impurity at most $1-1/K$, both attained by the uniform distribution.
Show also that both are $0$ exactly for pure nodes.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Gini: minimise $\sum_kp_k^2$ subject to $\sum_kp_k=1$ (Lagrange or Cauchy–Schwarz). Entropy: Jensen on the concave $\log$, or the Gibbs inequality.

</details>

<details>
<summary><b>✅ Solution</b></summary>

**Gini**: by Cauchy–Schwarz $1=(\sum_kp_k)^2\le K\sum_kp_k^2$, so $\sum p_k^2\ge1/K$ and Gini $\le1-1/K$, with equality iff all $p_k=1/K$.
**Entropy**: $H=\sum_kp_k\log_2\frac1{p_k}\le\log_2\sum_kp_k\frac1{p_k}=\log_2K$ by Jensen (concave $\log$), equality iff $1/p_k$ is constant.
**Zero**: both are sums of non-negative terms $p_k\log\frac1{p_k}$ and $p_k(1-p_k)$, which vanish only if each $p_k\in\{0,1\}$, i.e. the node is pure.
For two classes: $H\le1$ bit, Gini $\le0.5$ (see [[Information Theory]]).

</details>

## Part C · Code from scratch

### Exercise 11 · Impurity functions
*💻 Code · ★☆☆*

Implement `entropy(y)` (bits) and `gini(y)` taking an array of integer labels. Test on a few label arrays.

In [ ]:
def entropy(y):
    # TODO
    return None

def gini(y):
    # TODO
    return None

tests = [np.array([0, 0, 1, 1]), np.array([0, 1, 2, 2, 2, 1]), np.array([3, 3, 3])]

for _t in tests:
    _p = np.bincount(_t) / len(_t)
    check(f'entropy {list(_t)}', entropy(_t), sp_entropy(_p, base=2))
    check(f'gini {list(_t)}', gini(_t), 1 - np.sum(_p**2))

<details>
<summary><b>💡 Hint</b></summary>

`p = np.bincount(y) / len(y)`; drop zeros before taking logs.

</details>

<details>
<summary><b>✅ Solution</b></summary>

A pure node gives $0$ for both; the 50/50 node gives $1$ bit and $0.5$. Dropping $p_k=0$ implements the convention $0\log0=0$.

```python
def entropy(y):
    p = np.bincount(y) / len(y)
    p = p[p > 0]
    return float(-np.sum(p * np.log2(p)))

def gini(y):
    p = np.bincount(y) / len(y)
    return float(1 - np.sum(p ** 2))

tests = [np.array([0, 0, 1, 1]), np.array([0, 1, 2, 2, 2, 1]), np.array([3, 3, 3])]
```

</details>

### Exercise 12 · Exhaustive best split
*💻 Code · ★★☆*

Implement `best_split(X, y)` that tries every feature and every midpoint between consecutive distinct sorted values and returns
`(feature, threshold, gini_decrease)` for the split with the largest weighted Gini decrease. Compare with a depth-1 sklearn tree on the training data.

In [ ]:
def best_split(X, y):
    # TODO
    return None

split_root = best_split(Xtr, ytr)

_t = DecisionTreeClassifier(max_depth=1, random_state=0).fit(Xtr, ytr).tree_
_n = _t.n_node_samples
_dec = _t.impurity[0] - (_n[1]*_t.impurity[1] + _n[2]*_t.impurity[2]) / _n[0]
if split_root is None:
    check('root split', None, 0)
else:
    check('feature', split_root[0], _t.feature[0])
    check('threshold', split_root[1], _t.threshold[0], tol=1e-3)
    check('Gini decrease', split_root[2], _dec)

<details>
<summary><b>💡 Hint 1</b></summary>

For one feature, sort the values and loop over cut positions $i$; the left child is `y_sorted[:i]`. Only cut where `x_sorted[i-1] < x_sorted[i]`.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

To be fast, use cumulative class counts: `np.cumsum(np.eye(K)[y_sorted], axis=0)` gives the left-child counts for every cut at once.

</details>

<details>
<summary><b>✅ Solution</b></summary>

With cumulative counts each feature costs $O(N\log N)$ (the sort), so a split search is $O(dN\log N)$. The best root split is on one of the 'worst' (largest) radius/perimeter/concave-points features.

```python
def best_split(X, y):
    n, d = X.shape
    K = y.max() + 1
    parent = 1 - np.sum((np.bincount(y, minlength=K) / n) ** 2)
    best = (None, None, -np.inf)
    for j in range(d):
        order = np.argsort(X[:, j], kind='stable')
        xs, ys = X[order, j], y[order]
        left = np.cumsum(np.eye(K)[ys], axis=0)[:-1]          # counts for cut after position i
        right = left[-1] + np.eye(K)[ys[-1]] - left
        nl = np.arange(1, n); nr = n - nl
        g = (nl * (1 - np.sum((left / nl[:, None]) ** 2, 1)) + nr * (1 - np.sum((right / nr[:, None]) ** 2, 1))) / n
        valid = xs[1:] > xs[:-1]
        if not valid.any():
            continue
        i = np.argmin(np.where(valid, g, np.inf))
        if parent - g[i] > best[2] + 1e-12:
            best = (j, (xs[i] + xs[i + 1]) / 2, parent - g[i])
    return best

split_root = best_split(Xtr, ytr)
print(split_root)
```

</details>

### Exercise 13 · A recursive CART classifier
*💻 Code · ★★★*

Using `best_split`, implement `grow(X, y, depth, max_depth, min_samples)` returning a nested dict (leaf: `{'leaf': class}`;
node: `{'f', 't', 'left', 'right'}`, left = `x[f] <= t`) and `predict_tree(node, X)`. Grow to `max_depth=4` on the training data
and compare test accuracy with `DecisionTreeClassifier(max_depth=4)`.

In [ ]:
def grow(X, y, depth=0, max_depth=4, min_samples=2):
    # TODO
    return None

def predict_tree(node, X):
    # TODO
    return None

tree = grow(Xtr, ytr)
pred_tree = None if tree is None else predict_tree(tree, Xte)

_sk = DecisionTreeClassifier(max_depth=4, random_state=0).fit(Xtr, ytr)
if pred_tree is not None:
    print('my test acc', np.mean(pred_tree == yte), '| sklearn', _sk.score(Xte, yte))
check('agrees with sklearn on >= 97% of test points', None if pred_tree is None else np.mean(pred_tree == _sk.predict(Xte)) >= 0.97, True)

<details>
<summary><b>💡 Hint</b></summary>

Stop (make a leaf with the majority class) when depth = max_depth, the node is pure, it has fewer than `min_samples` points, or no valid split exists.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The recursion mirrors the definition: split, then grow each child. Small differences from sklearn can only come from ties between equally good splits.
Real implementations store the tree in flat arrays (as `tree_` does) for speed.

```python
def grow(X, y, depth=0, max_depth=4, min_samples=2):
    majority = int(np.bincount(y).argmax())
    if depth == max_depth or len(y) < min_samples or np.all(y == y[0]):
        return {'leaf': majority}
    f, t, dec = best_split(X, y)
    if f is None:
        return {'leaf': majority}
    m = X[:, f] <= t
    return {'f': f, 't': t,
            'left': grow(X[m], y[m], depth + 1, max_depth, min_samples),
            'right': grow(X[~m], y[~m], depth + 1, max_depth, min_samples)}

def predict_tree(node, X):
    out = np.empty(len(X), dtype=int)
    for i, x in enumerate(X):
        n = node
        while 'leaf' not in n:
            n = n['left'] if x[n['f']] <= n['t'] else n['right']
        out[i] = n['leaf']
    return out

tree = grow(Xtr, ytr)
pred_tree = predict_tree(tree, Xte)
```

</details>

### Exercise 14 · Depth vs overfitting
*💻 Code · ★☆☆*

Fit `DecisionTreeClassifier(max_depth=d, random_state=0)` for $d=1,\dots,12$ and `None` on the training split. Record train and test accuracy.
Store the train accuracy of the fully grown tree in `train_acc_full` and the depth with the best **5-fold CV accuracy on the training set** in `best_depth`.

In [ ]:
depths = list(range(1, 13))
train_acc_full = None
best_depth = None

check('fully grown tree memorises the training set', train_acc_full, 1.0)
_cv = [cross_val_score(DecisionTreeClassifier(max_depth=d, random_state=0), Xtr, ytr, cv=5).mean() for d in depths]
check('best depth by CV', best_depth, depths[int(np.argmax(_cv))])

<details>
<summary><b>💡 Hint</b></summary>

Training accuracy is a useless guide here; use `cross_val_score(..., cv=5)` on the training data for the choice.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Training accuracy climbs to 1.0 (every leaf pure) while test accuracy peaks at a small depth and then flattens or drops: variance takes over.
CV picks a shallow tree (a few levels). Use `min_samples_leaf` or pruning for a smoother control than depth.

```python
depths = list(range(1, 13))
for d in depths + [None]:
    m = DecisionTreeClassifier(max_depth=d, random_state=0).fit(Xtr, ytr)
    print(f'depth={d}: train {m.score(Xtr, ytr):.3f}  test {m.score(Xte, yte):.3f}')
train_acc_full = DecisionTreeClassifier(random_state=0).fit(Xtr, ytr).score(Xtr, ytr)
cv = [cross_val_score(DecisionTreeClassifier(max_depth=d, random_state=0), Xtr, ytr, cv=5).mean() for d in depths]
best_depth = depths[int(np.argmax(cv))]
print('best depth', best_depth)
```

</details>

### Exercise 15 · Cost-complexity pruning in practice
*💻 Code · ★★☆*

Use `DecisionTreeClassifier(random_state=0).cost_complexity_pruning_path(Xtr, ytr)` to get the candidate $\alpha$ values. Choose $\alpha$ by 5-fold CV
on the training data (`ccp_alpha=...`), refit, and report the number of leaves and test accuracy versus the unpruned tree. Store `best_alpha` and `n_leaves_pruned`.

In [ ]:
best_alpha = None
n_leaves_pruned = None

_path = DecisionTreeClassifier(random_state=0).cost_complexity_pruning_path(Xtr, ytr).ccp_alphas[:-1]
_cv = [cross_val_score(DecisionTreeClassifier(random_state=0, ccp_alpha=a), Xtr, ytr, cv=5).mean() for a in _path]
check('best alpha', best_alpha, _path[int(np.argmax(_cv))], tol=1e-9)
check('pruned tree has fewer leaves', None if n_leaves_pruned is None else n_leaves_pruned < DecisionTreeClassifier(random_state=0).fit(Xtr, ytr).get_n_leaves(), True)

<details>
<summary><b>💡 Hint</b></summary>

Drop the last $\alpha$ of the path (it prunes down to the root). `model.get_n_leaves()` counts leaves.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The pruned tree has half the leaves (16 → 8) with similar test accuracy (0.935 → 0.923; on 169 test points one point is 0.6%). A simpler tree with the same accuracy is easier to read and more stable. Each $\alpha$ on the path is a weakest-link value
from Exercise 9.

```python
path = DecisionTreeClassifier(random_state=0).cost_complexity_pruning_path(Xtr, ytr).ccp_alphas[:-1]
cv = [cross_val_score(DecisionTreeClassifier(random_state=0, ccp_alpha=a), Xtr, ytr, cv=5).mean() for a in path]
best_alpha = path[int(np.argmax(cv))]
full = DecisionTreeClassifier(random_state=0).fit(Xtr, ytr)
pruned = DecisionTreeClassifier(random_state=0, ccp_alpha=best_alpha).fit(Xtr, ytr)
n_leaves_pruned = pruned.get_n_leaves()
print(f'alpha={best_alpha:.4f}: leaves {full.get_n_leaves()} -> {n_leaves_pruned}, test {full.score(Xte, yte):.3f} -> {pruned.score(Xte, yte):.3f}')
```

</details>

### Exercise 16 · Trees are unstable
*💻 Code · ★★☆*

Train 30 fully grown trees on bootstrap resamples of the training data. (a) Record the root feature of each tree. (b) Measure the average pairwise
disagreement of their test predictions, and compare with the disagreement between two **bagged** ensembles of 15 trees each (majority vote).
Store `n_root_features` (number of distinct root features), `dis_trees` and `dis_bagged`.

In [ ]:
n_root_features = None
dis_trees = None
dis_bagged = None

check('averaging reduces disagreement', None if dis_trees is None else dis_bagged < dis_trees, True)

<details>
<summary><b>💡 Hint</b></summary>

Bootstrap: `idx = r.integers(0, n, n)`. Root feature: `model.tree_.feature[0]`. Disagreement of two prediction vectors: `np.mean(p1 != p2)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Several different root features appear across resamples, and two single trees disagree on several percent of test points; two bagged ensembles disagree far less.
This variance is exactly what bagging and random forests average away ([[Ensemble Methods]]).

```python
r = np.random.default_rng(1)
n = len(ytr)
roots, preds = [], []
for b in range(30):
    idx = r.integers(0, n, n)
    m = DecisionTreeClassifier(random_state=0).fit(Xtr[idx], ytr[idx])
    roots.append(m.tree_.feature[0]); preds.append(m.predict(Xte))
preds = np.array(preds)
n_root_features = len(set(roots))
dis_trees = np.mean([np.mean(preds[i] != preds[j]) for i in range(30) for j in range(i + 1, 30)])
vote = lambda P: (P.mean(0) > 0.5).astype(int)
dis_bagged = np.mean(vote(preds[:15]) != vote(preds[15:]))
print('distinct roots', n_root_features, '| single trees', round(dis_trees, 3), '| bagged', round(dis_bagged, 3))
```

</details>

### Exercise 17 · Axis-aligned staircases
*💻 Code · ★★☆*

Sample 1000 points uniformly in $[0,1]^2$ with label $y=[x_1>x_2]$. Fit a fully grown tree on $(x_1,x_2)$ and another on the single rotated feature $x_1-x_2$.
Store both leaf counts (`leaves_raw`, `leaves_rot`). What does this suggest about feature engineering for trees?

In [ ]:
leaves_raw = None
leaves_rot = None

check('rotated feature needs only 2 leaves', leaves_rot, 2)
check('raw features need a staircase (> 10 leaves)', None if leaves_raw is None else leaves_raw > 10, True)

<details>
<summary><b>💡 Hint</b></summary>

`DecisionTreeClassifier(random_state=0).fit(X, y).get_n_leaves()`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

On raw features the diagonal boundary is approximated by a staircase of dozens of leaves (and still generalises imperfectly between them).
One rotated feature makes the problem a single threshold. Adding linear combinations (or using oblique trees / PCA-rotated features) can help trees a lot,
while linear models get diagonal boundaries for free.

```python
r = np.random.default_rng(2)
Xd = r.uniform(size=(1000, 2))
yd = (Xd[:, 0] > Xd[:, 1]).astype(int)
leaves_raw = DecisionTreeClassifier(random_state=0).fit(Xd, yd).get_n_leaves()
leaves_rot = DecisionTreeClassifier(random_state=0).fit((Xd[:, 0] - Xd[:, 1])[:, None], yd).get_n_leaves()
print(leaves_raw, leaves_rot)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Decision Trees](Decision%20Trees.md).*